# Build your own animal for morphing_birds

Use this notebook before the DMD workshop notebook when you have custom keypoint data shaped `(n_frames, n_markers, 3)`. The aim is to define the marker order, the display polygons, the analysis markers, and a first `Animal3D` object that animates correctly.


In [ ]:
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "git+https://github.com/LydiaFrance/morphing_birds.git@custom-skeleton",
    ])
else:
    repo_root = Path.cwd()
    if not (repo_root / "src" / "morphing_birds").exists() and (repo_root.parent / "src" / "morphing_birds").exists():
        repo_root = repo_root.parent
    src_path = repo_root / "src"
    if src_path.exists():
        sys.path.insert(0, str(src_path))


In [ ]:
import numpy as np
import plotly.io as pio

from morphing_birds import Animal3D, SkeletonDefinition, animate_plotly

if IN_COLAB:
    pio.renderers.default = "colab"

np.set_printoptions(precision=3, suppress=True)


## 1. Start with data shaped `(frames, markers, xyz)`

Replace this synthetic example with your own loaded array. The important rule is that axis 1 must match `marker_names` exactly.


In [ ]:
marker_names = [
    "nose",
    "spine_front",
    "spine_mid",
    "hip",
    "tail_tip",
    "left_front_foot",
    "right_front_foot",
    "left_back_foot",
    "right_back_foot",
    "tracking_tag",
]

rest_pose = np.array(
    [
        [0.0, 1.2, 0.1],
        [0.0, 0.7, 0.0],
        [0.0, 0.1, 0.0],
        [0.0, -0.5, 0.0],
        [0.0, -1.1, 0.05],
        [-0.6, 0.45, -0.4],
        [0.6, 0.45, -0.4],
        [-0.5, -0.45, -0.4],
        [0.5, -0.45, -0.4],
        [0.0, 0.2, 0.35],
    ],
    dtype=float,
)

n_frames = 60
phase = np.linspace(0, 2 * np.pi, n_frames, endpoint=False)
motion = np.repeat(rest_pose[None, :, :], n_frames, axis=0)

left_step = np.sin(phase)
right_step = np.sin(phase + np.pi)
motion[:, marker_names.index("left_front_foot"), 2] += 0.20 * np.maximum(left_step, 0)
motion[:, marker_names.index("right_back_foot"), 2] += 0.18 * np.maximum(left_step, 0)
motion[:, marker_names.index("right_front_foot"), 2] += 0.20 * np.maximum(right_step, 0)
motion[:, marker_names.index("left_back_foot"), 2] += 0.18 * np.maximum(right_step, 0)
motion[:, marker_names.index("tail_tip"), 0] += 0.12 * np.sin(phase)
motion[:, marker_names.index("spine_mid"), 2] += 0.04 * np.sin(2 * phase)

assert motion.shape == (n_frames, len(marker_names), 3)
motion.shape


## 2. Define the skeleton

`body_sections` are polygons or line strips for display. `analysis_exclude` markers are displayed but skipped by PCA/SVD/FFT/DMD, so only put fixed landmarks there. Left/right pairs and centre markers make symmetry helpers explicit even when your names do not use the builtin bird conventions.


In [ ]:
body_sections = {
    "body": ["nose", "spine_front", "spine_mid", "hip", "tail_tip"],
    "front_support": ["left_front_foot", "spine_front", "right_front_foot"],
    "back_support": ["left_back_foot", "hip", "right_back_foot"],
    "left_side": ["left_front_foot", "spine_mid", "left_back_foot"],
    "right_side": ["right_front_foot", "spine_mid", "right_back_foot"],
}

marker_pairs = [
    ("left_front_foot", "right_front_foot"),
    ("left_back_foot", "right_back_foot"),
]

centre_markers = ["nose", "spine_front", "spine_mid", "hip", "tail_tip", "tracking_tag"]

skel = SkeletonDefinition.from_markers(
    "workshop_animal",
    marker_names,
    body_sections=body_sections,
    analysis_exclude=["tracking_tag"],
    marker_pairs=marker_pairs,
    centre_markers=centre_markers,
)

print("All markers:", skel.all_marker_names)
print("Analysis markers:", skel.analysis_markers)
print("Display-only markers:", skel.display_only_markers)


## 3. Build the animal and check the raw animation

`rest_pose` is the static shape used for display-only markers and for the first view of the animal.


In [ ]:
rest_pose = np.nanmean(motion, axis=0)
animal = Animal3D(skel, data=rest_pose)

fig = animate_plotly(animal, motion, axes_visible=False)
fig.show()


## 4. Handoff to the DMD notebook

The maths should use only the analysis subset. Reconstructed analysis data can be animated directly against the same `animal`.


In [ ]:
X = animal.get_analysis_data(motion)
X_flat = X.reshape(X.shape[0], -1)

# In the DMD notebook, replace this with your reconstructed flat array.
recon_flat = X_flat.copy()
recon = recon_flat.reshape(-1, X.shape[1], 3)

fig = animate_plotly(animal, recon, axes_visible=False)
fig.show()

print("DMD input:", X_flat.shape)
print("Animation reconstruction:", recon.shape)


## Copy this cell into the main DMD workshop notebook

Keep `marker_names`, `body_sections`, `marker_pairs`, `centre_markers`, `skel`, `rest_pose`, `animal`, and `motion` together. Replace only the synthetic `motion` block with your data loader.


In [ ]:
marker_names = [
    "nose",
    "spine_front",
    "spine_mid",
    "hip",
    "tail_tip",
    "left_front_foot",
    "right_front_foot",
    "left_back_foot",
    "right_back_foot",
    "tracking_tag",
]

rest_pose = np.array(
    [
        [0.0, 1.2, 0.1],
        [0.0, 0.7, 0.0],
        [0.0, 0.1, 0.0],
        [0.0, -0.5, 0.0],
        [0.0, -1.1, 0.05],
        [-0.6, 0.45, -0.4],
        [0.6, 0.45, -0.4],
        [-0.5, -0.45, -0.4],
        [0.5, -0.45, -0.4],
        [0.0, 0.2, 0.35],
    ],
    dtype=float,
)

# Replace this synthetic motion with your data loader.
motion = np.repeat(rest_pose[None, :, :], 60, axis=0)
assert motion.ndim == 3 and motion.shape[1:] == (len(marker_names), 3)

body_sections = {
    "body": ["nose", "spine_front", "spine_mid", "hip", "tail_tip"],
    "front_support": ["left_front_foot", "spine_front", "right_front_foot"],
    "back_support": ["left_back_foot", "hip", "right_back_foot"],
    "left_side": ["left_front_foot", "spine_mid", "left_back_foot"],
    "right_side": ["right_front_foot", "spine_mid", "right_back_foot"],
}
marker_pairs = [("left_front_foot", "right_front_foot"), ("left_back_foot", "right_back_foot")]
centre_markers = ["nose", "spine_front", "spine_mid", "hip", "tail_tip", "tracking_tag"]

skel = SkeletonDefinition.from_markers(
    "workshop_animal",
    marker_names,
    body_sections=body_sections,
    analysis_exclude=["tracking_tag"],
    marker_pairs=marker_pairs,
    centre_markers=centre_markers,
)
rest_pose = np.nanmean(motion, axis=0)
animal = Animal3D(skel, data=rest_pose)

X = animal.get_analysis_data(motion)
X_flat = X.reshape(X.shape[0], -1)


## Troubleshooting

- Marker order mismatch: `marker_names[i]` must name `motion[:, i, :]`.
- Wrong units or axes: check whether your data is metres, centimetres, pixels, or whether vertical is `z`.
- Missing values: use `NaN` for unknown coordinates so `np.nanmean` does the right thing.
- Polygon marker typo: every name in `body_sections` must also appear in `marker_names`.
- Display-only markers: put fixed landmarks in `analysis_exclude`; moving keypoints usually belong in the analysis set.
